## ResNet

In [17]:
import torch
import torchvision.models as models
from PIL import Image
import pandas as pd
from tqdm import tqdm
import time

# Check GPU is available (important for training speed!)
print(torch.cuda.is_available())  # ideally prints True

# Check ResNet loads
model = models.resnet18(pretrained=True)
print("ResNet loaded ok")

True
ResNet loaded ok


In [2]:
import torch

print(torch.cuda.is_available())        # should print True
print(torch.cuda.get_device_name(0))    # prints your GPU name
print(torch.cuda.mem_get_info())

True
NVIDIA GeForce RTX 4060 Laptop GPU
(7443841024, 8585216000)


In [14]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
import json
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split
import pandas as pd

# ── 1. DATASET ────────────────────────────────────────────────────────────────

class ShadowDataset(Dataset):
    def __init__(self, folder, img_size=224):
        self.folder = folder
        self.img_size = img_size
        self.transform = T.Compose([
            T.Resize((img_size, img_size)),
            T.ToTensor(),
            T.Normalize([0.485, 0.456, 0.406],  # ImageNet mean
                        [0.229, 0.224, 0.225])   # ImageNet std
        ])
        
        # Collect all images that have a matching json
        self.samples = [
            f.replace(".png", "")
            for f in os.listdir(folder)
            if f.endswith(".png") and 
               os.path.exists(os.path.join(folder, f.replace(".png", ".json")))
        ]
        print(f"Found {len(self.samples)} samples in {folder}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        name = self.samples[idx]
        
        # ── Load image ──
        img_path = os.path.join(self.folder, f"{name}.png")
        img = Image.open(img_path).convert("RGB")
        W, H = img.size          # original size, needed for normalizing
        img = self.transform(img)
        
        # ── Load annotation ──
        json_path = os.path.join(self.folder, f"{name}.json")
        with open(json_path) as f:
            ann = json.load(f)
        
        # Normalize all corner coordinates to 0-1
        # Note: values CAN be > 1.0 since person is off-screen!
        bbox = ann["bbox"]
        corners = torch.tensor([
            bbox["top_left"][0]     / W,
            bbox["top_left"][1]     / H,
            bbox["top_right"][0]    / W,
            bbox["top_right"][1]    / H,
            bbox["bottom_left"][0]  / W,
            bbox["bottom_left"][1]  / H,
            bbox["bottom_right"][0] / W,
            bbox["bottom_right"][1] / H,
        ], dtype=torch.float32)

        direction = torch.tensor(
            [ann["walking_into_frame_bool"]], 
        dtype=torch.float32
)
        
        return img, corners, direction, W, H, name




In [15]:
class AttentionBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.att = nn.Sequential(
            nn.Conv2d(channels, channels // 8, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(channels // 8, channels, kernel_size=1),
            nn.Sigmoid()    # outputs 0-1 weight per spatial location
        )
    
    def forward(self, x):
        return x * self.att(x)   # suppress irrelevant areas


class ShadowDetector(nn.Module):
    def __init__(self):
        super().__init__()
        
        # V2: Pretrained ResNet18 backbone (remove final pool + fc)
        resnet = models.resnet18(pretrained=True)
        self.backbone = nn.Sequential(*list(resnet.children())[:-2])
        # Output: [batch, 512, 7, 7] for 224x224 input
        
        # Freeze first few layers (basic edges/textures already learned)
        for param in list(self.backbone.parameters())[:6]:
            param.requires_grad = False
        
        # V4: Attention on backbone output
        self.attention = AttentionBlock(512)
        
        self.pool = nn.AdaptiveAvgPool2d(1)   # [batch, 512, 1, 1]
        
        # Bounding box head → 8 values (4 corners × x,y)
        self.box_head = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 8)
            # No sigmoid! corners can be outside 0-1 (off-screen)
        )
        
        # Direction head → 1 value (0 or 1)
        self.direction_head = nn.Sequential(
            nn.Linear(512, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()    # sigmoid is fine, output is always 0 or 1
        )

    def forward(self, x):
        features = self.backbone(x)          # [batch, 512, 7, 7]
        features = self.attention(features)  # [batch, 512, 7, 7]
        pooled   = self.pool(features)       # [batch, 512, 1, 1]
        pooled   = pooled.view(pooled.size(0), -1)  # [batch, 512]
        
        corners   = self.box_head(pooled)       # [batch, 8]
        direction = self.direction_head(pooled) # [batch, 1]
        
        return corners, direction

In [19]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

# Load dataset
dataset = ShadowDataset("data/train_data/train_data")

# 80/20 split
train_size = int(0.8 * len(dataset))
val_size   = len(dataset) - train_size
train_set, val_set = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=32)

# Model, optimizer, losses
model     = ShadowDetector().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)
box_loss  = nn.MSELoss()
dir_loss  = nn.BCELoss()

best_val_loss = float("inf")

for epoch in range(50):
    start = time.time()
    
    # ── Train ──
    model.train()
    train_losses = []
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1:02d}/50 [Train]", leave=False)
    for imgs, corners, directions, W, H, _ in loop:
        imgs       = imgs.to(device)
        corners    = corners.to(device)
        directions = directions.to(device)
        
        optimizer.zero_grad()
        pred_corners, pred_dir = model(imgs)
        
        loss = box_loss(pred_corners, corners) * 10 + \
               dir_loss(pred_dir, directions)
        
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())
        loop.set_postfix(loss=f"{loss.item():.4f}")  # live loss in bar
    
    # ── Validate ──
    model.eval()
    val_losses = []
    loop = tqdm(val_loader, desc=f"Epoch {epoch+1:02d}/50 [Val]  ", leave=False)
    with torch.no_grad():
        for imgs, corners, directions, W, H, _ in loop:
            imgs       = imgs.to(device)
            corners    = corners.to(device)
            directions = directions.to(device)
            
            pred_corners, pred_dir = model(imgs)
            loss = box_loss(pred_corners, corners)
            val_losses.append(loss.item())
            loop.set_postfix(loss=f"{loss.item():.4f}")
    
    avg_train = sum(train_losses) / len(train_losses)
    avg_val   = sum(val_losses)   / len(val_losses)
    elapsed   = time.time() - start
    
    print(f"Epoch {epoch+1:02d}/50 | Train: {avg_train:.4f} | Val: {avg_val:.4f} | Time: {elapsed:.1f}s")
    
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save(model.state_dict(), "best_model.pth")
        print(f"           ↑ saved new best (val={avg_val:.4f})")
    
    scheduler.step()
    
    # Save best model
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save(model.state_dict(), "best_model.pth")
        print(f"           ↑ saved new best model")
    
    scheduler.step()



Using: cuda
Found 1692 samples in data/train_data/train_data


Epoch 01/50 | Train: 1.3455 | Val: 0.0124 | Time: 43.6s
           ↑ saved new best (val=0.0124)


Epoch 02/50 | Train: 0.8780 | Val: 0.0055 | Time: 34.1s
           ↑ saved new best (val=0.0055)


Epoch 03/50 | Train: 0.8336 | Val: 0.0019 | Time: 33.2s
           ↑ saved new best (val=0.0019)


Epoch 04/50 | Train: 0.8146 | Val: 0.0029 | Time: 33.5s


...

Epoch 46/50 | Train: 0.0806 | Val: 0.0011 | Time: 32.9s


Epoch 47/50 | Train: 0.0846 | Val: 0.0011 | Time: 32.7s


Epoch 48/50 | Train: 0.0865 | Val: 0.0012 | Time: 32.6s


Epoch 49/50 | Train: 0.0732 | Val: 0.0013 | Time: 32.4s


Epoch 50/50 | Train: 0.0719 | Val: 0.0011 | Time: 32.7s


In [20]:
# Load best model
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

iou_scores = []
dir_correct = 0
dir_total = 0

with torch.no_grad():
    for imgs, corners, directions, W, H, _ in val_loader:
        imgs       = imgs.to(device)
        corners    = corners.to(device)
        directions = directions.to(device)

        pred_corners, pred_dir = model(imgs)

        W = W.float().to(device).unsqueeze(1)
        H = H.float().to(device).unsqueeze(1)
        scale = torch.cat([W, H], dim=1).repeat(1, 4)

        pred_px = (pred_corners * scale).view(-1, 4, 2)
        true_px = (corners      * scale).view(-1, 4, 2)

        pred_xmin = pred_px[:, :, 0].min(dim=1).values
        pred_ymin = pred_px[:, :, 1].min(dim=1).values
        pred_xmax = pred_px[:, :, 0].max(dim=1).values
        pred_ymax = pred_px[:, :, 1].max(dim=1).values

        true_xmin = true_px[:, :, 0].min(dim=1).values
        true_ymin = true_px[:, :, 1].min(dim=1).values
        true_xmax = true_px[:, :, 0].max(dim=1).values
        true_ymax = true_px[:, :, 1].max(dim=1).values

        inter_xmin = torch.max(pred_xmin, true_xmin)
        inter_ymin = torch.max(pred_ymin, true_ymin)
        inter_xmax = torch.min(pred_xmax, true_xmax)
        inter_ymax = torch.min(pred_ymax, true_ymax)

        inter_w = (inter_xmax - inter_xmin).clamp(min=0)
        inter_h = (inter_ymax - inter_ymin).clamp(min=0)
        intersection = inter_w * inter_h

        pred_area = (pred_xmax - pred_xmin) * (pred_ymax - pred_ymin)
        true_area = (true_xmax - true_xmin) * (true_ymax - true_ymin)
        union = pred_area + true_area - intersection

        iou = intersection / union.clamp(min=1e-6)
        iou_scores.append(iou.mean().item())

        # Direction accuracy
        pred_label = (pred_dir > 0.5).float()
        dir_correct += (pred_label == directions).sum().item()
        dir_total   += directions.size(0)

avg_iou  = sum(iou_scores) / len(iou_scores)
dir_acc  = dir_correct / dir_total

print(f"── Final Validation Results ──")
print(f"Avg IoU:        {avg_iou:.3f}  (expected competition score)")
print(f"Direction Acc:  {dir_acc*100:.1f}%")
print(f"Est. Score:     {avg_iou + (dir_acc - 0.5) * 0.1:.3f}  (rough estimate with direction bonus)")

── Final Validation Results ──
Avg IoU:        0.571  (expected competition score)
Direction Acc:  49.3%
Est. Score:     0.570  (rough estimate with direction bonus)


In [ ]:
def generate_submission(model, test_folder, output_path="submission.csv"):
    model.eval()
    
    transform = T.Compose([
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    results = []
    
    for fname in sorted(os.listdir(test_folder)):
        if not fname.endswith(".png"):
            continue
        
        img = Image.open(os.path.join(test_folder, fname)).convert("RGB")
        W, H = img.size
        
        inp = transform(img).unsqueeze(0).to(device)
        
        with torch.no_grad():
            corners, direction = model(inp)
        
        c = corners[0].cpu().tolist()
        
        # Convert 4 corners back to pixel coords
        x_coords = [c[0]*W, c[2]*W, c[4]*W, c[6]*W]
        y_coords = [c[1]*H, c[3]*H, c[5]*H, c[7]*H]
        
        # Derive bounding box from corners
        xmin = min(x_coords)
        xmax = max(x_coords)
        ymin = min(y_coords)
        ymax = max(y_coords)
        
        results.append({
            "xmin":      xmin,
            "ymin":      ymin,
            "xmax":      xmax,
            "ymax":      ymax,
            "direction": round(direction[0].item())  # 0 or 1
        })
    
    df = pd.DataFrame(results)
    df.to_csv(output_path, index=False)
    print(f"Saved {len(results)} predictions to {output_path}")

In [ ]:
model.load_state_dict(torch.load("best_model.pth"))
generate_submission(model, "data/test_data/test_data")

In [23]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
import json
import os
import time
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split
import pandas as pd
from tqdm import tqdm

# ── 1. DATASET ────────────────────────────────────────────────────────────────

class ShadowDataset(Dataset):
    def __init__(self, folder, img_size=224):
        self.folder = folder
        self.img_size = img_size
        self.transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406],
                [0.229, 0.224, 0.225])
    # no augmentation - test images look identical to train
])
        
        self.samples = [
            f.replace(".png", "")
            for f in os.listdir(folder)
            if f.endswith(".png") and 
               os.path.exists(os.path.join(folder, f.replace(".png", ".json")))
        ]
        print(f"Found {len(self.samples)} samples in {folder}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        name = self.samples[idx]
        
        # ── Load image ──
        img_path = os.path.join(self.folder, f"{name}.png")
        img = Image.open(img_path).convert("RGB")
        W, H = img.size
        img = self.transform(img)
        
        # ── Load annotation ──
        json_path = os.path.join(self.folder, f"{name}.json")
        with open(json_path) as f:
            ann = json.load(f)
        
        bbox = ann["bbox"]
        corners = torch.tensor([
            bbox["top_left"][0]     / W,
            bbox["top_left"][1]     / H,
            bbox["top_right"][0]    / W,
            bbox["top_right"][1]    / H,
            bbox["bottom_left"][0]  / W,
            bbox["bottom_left"][1]  / H,
            bbox["bottom_right"][0] / W,
            bbox["bottom_right"][1] / H,
        ], dtype=torch.float32)
        
        direction = torch.tensor(
            [ann["walking_into_frame_bool"]], 
            dtype=torch.float32
        )
        
        return img, corners, direction, W, H, name


# ── 2. MODEL ──────────────────────────────────────────────────────────────────

class AttentionBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.att = nn.Sequential(
            nn.Conv2d(channels, channels // 8, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(channels // 8, channels, kernel_size=1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return x * self.att(x)


class ShadowDetector(nn.Module):
    def __init__(self):
        super().__init__()
        
        # Upgraded to ResNet50
        resnet = models.resnet50(pretrained=True)
        self.backbone = nn.Sequential(*list(resnet.children())[:-2])
        # Output: [batch, 2048, 7, 7] for 224x224 input
        
        # Freeze first few layers
        for param in list(self.backbone.parameters())[:6]:
            param.requires_grad = False
        
        # Attention on 2048 channels (resnet50 output)
        self.attention = AttentionBlock(2048)
        
        self.pool = nn.AdaptiveAvgPool2d(1)
        
        # Box head
        self.box_head = nn.Sequential(
            nn.Linear(2048, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 8)
        )
        
        # Direction head
        self.direction_head = nn.Sequential(
            nn.Linear(2048, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        features = self.backbone(x)
        features = self.attention(features)
        pooled   = self.pool(features)
        pooled   = pooled.view(pooled.size(0), -1)
        
        corners   = self.box_head(pooled)
        direction = self.direction_head(pooled)
        
        return corners, direction


# ── 3. IoU LOSS ───────────────────────────────────────────────────────────────

def iou_loss(pred, target, W, H):
    scale = torch.cat([W, H], dim=1).repeat(1, 4)
    pred_px = (pred * scale).view(-1, 4, 2)
    true_px = (target * scale).view(-1, 4, 2)

    pred_xmin = pred_px[:, :, 0].min(dim=1).values
    pred_ymin = pred_px[:, :, 1].min(dim=1).values
    pred_xmax = pred_px[:, :, 0].max(dim=1).values
    pred_ymax = pred_px[:, :, 1].max(dim=1).values

    true_xmin = true_px[:, :, 0].min(dim=1).values
    true_ymin = true_px[:, :, 1].min(dim=1).values
    true_xmax = true_px[:, :, 0].max(dim=1).values
    true_ymax = true_px[:, :, 1].max(dim=1).values

    inter_xmin = torch.max(pred_xmin, true_xmin)
    inter_ymin = torch.max(pred_ymin, true_ymin)
    inter_xmax = torch.min(pred_xmax, true_xmax)
    inter_ymax = torch.min(pred_ymax, true_ymax)

    inter_w = (inter_xmax - inter_xmin).clamp(min=0)
    inter_h = (inter_ymax - inter_ymin).clamp(min=0)
    intersection = inter_w * inter_h

    pred_area = (pred_xmax - pred_xmin) * (pred_ymax - pred_ymin)
    true_area = (true_xmax - true_xmin) * (true_ymax - true_ymin)
    union = pred_area + true_area - intersection

    iou = intersection / union.clamp(min=1e-6)
    return 1 - iou.mean()


# ── 4. TRAINING ───────────────────────────────────────────────────────────────

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

dataset = ShadowDataset("data/train_data/train_data")

train_size = int(0.8 * len(dataset))
val_size   = len(dataset) - train_size
train_set, val_set = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=32)

model     = ShadowDetector().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)
dir_loss  = nn.BCELoss()

best_val_iou = 0.0

for epoch in range(50):
    start = time.time()
    
    # Use MSE for first 10 epochs to get boxes in the right area
    # Then switch to IoU loss to fine tune overlap
    use_iou_loss = epoch >= 10

    # ── Train ──
    model.train()
    train_losses = []
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1:02d}/50 [Train]", leave=False)
    for imgs, corners, directions, W, H, _ in loop:
        imgs       = imgs.to(device)
        corners    = corners.to(device)
        directions = directions.to(device)
        W_s = W.float().to(device).unsqueeze(1)
        H_s = H.float().to(device).unsqueeze(1)
        
        optimizer.zero_grad()
        pred_corners, pred_dir = model(imgs)
        
        if use_iou_loss:
            box_l = iou_loss(pred_corners, corners, W_s, H_s)
        else:
            box_l = nn.MSELoss()(pred_corners, corners)

        loss = box_l * 10 + dir_loss(pred_dir, directions)
        
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())
        loop.set_postfix(loss=f"{loss.item():.4f}", mode="IoU" if use_iou_loss else "MSE")
    
    # ── Validate ──
    model.eval()
    val_losses  = []
    iou_scores  = []
    pixel_errors = []
    loop = tqdm(val_loader, desc=f"Epoch {epoch+1:02d}/50 [Val]  ", leave=False)
    with torch.no_grad():
        for imgs, corners, directions, W, H, _ in loop:
            imgs       = imgs.to(device)
            corners    = corners.to(device)
            directions = directions.to(device)
            W_s = W.float().to(device).unsqueeze(1)
            H_s = H.float().to(device).unsqueeze(1)
            
            pred_corners, pred_dir = model(imgs)

            loss = iou_loss(pred_corners, corners, W_s, H_s)
            val_losses.append(loss.item())
            loop.set_postfix(loss=f"{loss.item():.4f}")

            # Pixel error
            scale = torch.cat([W_s, H_s], dim=1).repeat(1, 4)
            pred_px = pred_corners * scale
            true_px = corners      * scale
            diff = (pred_px - true_px).view(-1, 4, 2)
            dist = torch.sqrt((diff ** 2).sum(dim=2))
            pixel_errors.append(dist.mean().item())

            # IoU
            pred_px = pred_px.view(-1, 4, 2)
            true_px = true_px.view(-1, 4, 2)

            pred_xmin = pred_px[:, :, 0].min(dim=1).values
            pred_ymin = pred_px[:, :, 1].min(dim=1).values
            pred_xmax = pred_px[:, :, 0].max(dim=1).values
            pred_ymax = pred_px[:, :, 1].max(dim=1).values

            true_xmin = true_px[:, :, 0].min(dim=1).values
            true_ymin = true_px[:, :, 1].min(dim=1).values
            true_xmax = true_px[:, :, 0].max(dim=1).values
            true_ymax = true_px[:, :, 1].max(dim=1).values

            inter_xmin = torch.max(pred_xmin, true_xmin)
            inter_ymin = torch.max(pred_ymin, true_ymin)
            inter_xmax = torch.min(pred_xmax, true_xmax)
            inter_ymax = torch.min(pred_ymax, true_ymax)

            inter_w = (inter_xmax - inter_xmin).clamp(min=0)
            inter_h = (inter_ymax - inter_ymin).clamp(min=0)
            intersection = inter_w * inter_h

            pred_area = (pred_xmax - pred_xmin) * (pred_ymax - pred_ymin)
            true_area = (true_xmax - true_xmin) * (true_ymax - true_ymin)
            union = pred_area + true_area - intersection
            iou = intersection / union.clamp(min=1e-6)
            iou_scores.append(iou.mean().item())

    avg_train  = sum(train_losses)  / len(train_losses)
    avg_val    = sum(val_losses)    / len(val_losses)
    avg_iou    = sum(iou_scores)    / len(iou_scores)
    avg_px_err = sum(pixel_errors)  / len(pixel_errors)
    elapsed    = time.time() - start

    print(f"Epoch {epoch+1:02d}/50 | Train: {avg_train:.4f} | Val: {avg_val:.4f} | IoU: {avg_iou:.3f} | Pixel Error: {avg_px_err:.1f}px | Time: {elapsed:.1f}s")

    # Save best model based on IoU now
    if avg_iou > best_val_iou:
        best_val_iou = avg_iou
        torch.save(model.state_dict(), "best_model.pth")
        print(f"           ↑ saved new best (IoU={avg_iou:.3f}, pixel_err={avg_px_err:.1f}px)")

    scheduler.step()


# ── 5. SUBMISSION ─────────────────────────────────────────────────────────────

def generate_submission(model, test_folder, output_path="submission.csv"):
    model.eval()
    
    transform = T.Compose([
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    results = []
    
    for fname in sorted(os.listdir(test_folder)):
        if not fname.endswith(".png"):
            continue
        
        img = Image.open(os.path.join(test_folder, fname)).convert("RGB")
        W, H = img.size
        
        inp = transform(img).unsqueeze(0).to(device)
        
        with torch.no_grad():
            corners, direction = model(inp)
        
        c = corners[0].cpu().tolist()
        
        x_coords = [c[0]*W, c[2]*W, c[4]*W, c[6]*W]
        y_coords = [c[1]*H, c[3]*H, c[5]*H, c[7]*H]
        
        xmin = min(x_coords)
        xmax = max(x_coords)
        ymin = min(y_coords)
        ymax = max(y_coords)

        # Abstain if not confident
        dir_prob = direction[0].item()
        if dir_prob > 0.75:
            dir_label = 1
        elif dir_prob < 0.25:
            dir_label = 0
        else:
            dir_label = -1

        results.append({
            "xmin":      xmin,
            "ymin":      ymin,
            "xmax":      xmax,
            "ymax":      ymax,
            "direction": dir_label
        })
    
    df = pd.DataFrame(results)
    df.to_csv(output_path, index=False)
    print(f"Saved {len(results)} predictions to {output_path}")

# Load best model and generate submission
model.load_state_dict(torch.load("best_model.pth"))
generate_submission(model, "data/test_data/test_data")


Using: cuda
Found 1692 samples in data/train_data/train_data


Epoch 01/50 | Train: 1.7179 | Val: 0.8243 | IoU: 0.176 | Pixel Error: 115.7px | Time: 24.5s
           ↑ saved new best (IoU=0.176, pixel_err=115.7px)


Epoch 02/50 | Train: 0.9639 | Val: 0.8996 | IoU: 0.100 | Pixel Error: 190.9px | Time: 25.7s


Epoch 03/50 | Train: 0.9262 | Val: 0.6986 | IoU: 0.301 | Pixel Error: 64.5px | Time: 25.3s
           ↑ saved new best (IoU=0.301, pixel_err=64.5px)


Epoch 04/50 | Train: 0.8532 | Val: 0.7485 | IoU: 0.251 | Pixel Error: 77.4px | Time: 25.2s


...

Epoch 10/50 | Train: 0.7972 | Val: 0.6196 | IoU: 0.380 | Pixel Error: 41.4px | Time: 25.1s


Epoch 11/50 | Train: 9.4710 | Val: 0.9229 | IoU: 0.077 | Pixel Error: 4468.6px | Time: 24.9s


Epoch 12/50 | Train: 9.1245 | Val: 0.7786 | IoU: 0.221 | Pixel Error: 538.1px | Time: 27.1s


Epoch 13/50 | Train: 9.0632 | Val: 0.8150 | IoU: 0.185 | Pixel Error: 544.2px | Time: 31.0s


KeyboardInterrupt: 

In [25]:
class ShadowDetector(nn.Module):
    def __init__(self):
        super().__init__()
        
        # Simple CNN - no pretrained weights needed for synthetic consistent images
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),  nn.ReLU(), nn.MaxPool2d(2),  # 112x112
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 56x56
            nn.Conv2d(64, 128, 3, padding=1),nn.ReLU(), nn.MaxPool2d(2),  # 28x28
            nn.Conv2d(128, 256, 3, padding=1),nn.ReLU(), nn.MaxPool2d(2), # 14x14
        )
        
        self.pool = nn.AdaptiveAvgPool2d(1)  # [batch, 256]
        
        self.box_head = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 8)
        )
        
        self.direction_head = nn.Sequential(
            nn.Linear(256, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        features = self.backbone(x)
        pooled   = self.pool(features)
        pooled   = pooled.view(pooled.size(0), -1)
        
        corners   = self.box_head(pooled)
        direction = self.direction_head(pooled)
        
        return corners, direction

In [26]:

def iou_loss(pred, target, W, H):
    scale = torch.cat([W, H], dim=1).repeat(1, 4)
    pred_px = (pred * scale).view(-1, 4, 2)
    true_px = (target * scale).view(-1, 4, 2)

    pred_xmin = pred_px[:, :, 0].min(dim=1).values
    pred_ymin = pred_px[:, :, 1].min(dim=1).values
    pred_xmax = pred_px[:, :, 0].max(dim=1).values
    pred_ymax = pred_px[:, :, 1].max(dim=1).values

    true_xmin = true_px[:, :, 0].min(dim=1).values
    true_ymin = true_px[:, :, 1].min(dim=1).values
    true_xmax = true_px[:, :, 0].max(dim=1).values
    true_ymax = true_px[:, :, 1].max(dim=1).values

    inter_xmin = torch.max(pred_xmin, true_xmin)
    inter_ymin = torch.max(pred_ymin, true_ymin)
    inter_xmax = torch.min(pred_xmax, true_xmax)
    inter_ymax = torch.min(pred_ymax, true_ymax)

    inter_w = (inter_xmax - inter_xmin).clamp(min=0)
    inter_h = (inter_ymax - inter_ymin).clamp(min=0)
    intersection = inter_w * inter_h

    pred_area = (pred_xmax - pred_xmin) * (pred_ymax - pred_ymin)
    true_area = (true_xmax - true_xmin) * (true_ymax - true_ymin)
    union = pred_area + true_area - intersection

    iou = intersection / union.clamp(min=1e-6)
    return 1 - iou.mean()


# ── 4. TRAINING ───────────────────────────────────────────────────────────────

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

dataset = ShadowDataset("data/train_data/train_data")

train_size = int(0.8 * len(dataset))
val_size   = len(dataset) - train_size
train_set, val_set = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=32)

model     = ShadowDetector().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)
dir_loss  = nn.BCELoss()

best_val_iou = 0.0

for epoch in range(50):
    start = time.time()
    
    # Use MSE for first 10 epochs to get boxes in the right area
    # Then switch to IoU loss to fine tune overlap
    use_iou_loss = epoch >= 10

    # ── Train ──
    model.train()
    train_losses = []
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1:02d}/50 [Train]", leave=False)
    for imgs, corners, directions, W, H, _ in loop:
        imgs       = imgs.to(device)
        corners    = corners.to(device)
        directions = directions.to(device)
        W_s = W.float().to(device).unsqueeze(1)
        H_s = H.float().to(device).unsqueeze(1)
        
        optimizer.zero_grad()
        pred_corners, pred_dir = model(imgs)
        
        if use_iou_loss:
            box_l = iou_loss(pred_corners, corners, W_s, H_s)
        else:
            box_l = nn.MSELoss()(pred_corners, corners)

        loss = box_l * 10 + dir_loss(pred_dir, directions)
        
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())
        loop.set_postfix(loss=f"{loss.item():.4f}", mode="IoU" if use_iou_loss else "MSE")
    
    # ── Validate ──
    model.eval()
    val_losses  = []
    iou_scores  = []
    pixel_errors = []
    loop = tqdm(val_loader, desc=f"Epoch {epoch+1:02d}/50 [Val]  ", leave=False)
    with torch.no_grad():
        for imgs, corners, directions, W, H, _ in loop:
            imgs       = imgs.to(device)
            corners    = corners.to(device)
            directions = directions.to(device)
            W_s = W.float().to(device).unsqueeze(1)
            H_s = H.float().to(device).unsqueeze(1)
            
            pred_corners, pred_dir = model(imgs)

            loss = iou_loss(pred_corners, corners, W_s, H_s)
            val_losses.append(loss.item())
            loop.set_postfix(loss=f"{loss.item():.4f}")

            # Pixel error
            scale = torch.cat([W_s, H_s], dim=1).repeat(1, 4)
            pred_px = pred_corners * scale
            true_px = corners      * scale
            diff = (pred_px - true_px).view(-1, 4, 2)
            dist = torch.sqrt((diff ** 2).sum(dim=2))
            pixel_errors.append(dist.mean().item())

            # IoU
            pred_px = pred_px.view(-1, 4, 2)
            true_px = true_px.view(-1, 4, 2)

            pred_xmin = pred_px[:, :, 0].min(dim=1).values
            pred_ymin = pred_px[:, :, 1].min(dim=1).values
            pred_xmax = pred_px[:, :, 0].max(dim=1).values
            pred_ymax = pred_px[:, :, 1].max(dim=1).values

            true_xmin = true_px[:, :, 0].min(dim=1).values
            true_ymin = true_px[:, :, 1].min(dim=1).values
            true_xmax = true_px[:, :, 0].max(dim=1).values
            true_ymax = true_px[:, :, 1].max(dim=1).values

            inter_xmin = torch.max(pred_xmin, true_xmin)
            inter_ymin = torch.max(pred_ymin, true_ymin)
            inter_xmax = torch.min(pred_xmax, true_xmax)
            inter_ymax = torch.min(pred_ymax, true_ymax)

            inter_w = (inter_xmax - inter_xmin).clamp(min=0)
            inter_h = (inter_ymax - inter_ymin).clamp(min=0)
            intersection = inter_w * inter_h

            pred_area = (pred_xmax - pred_xmin) * (pred_ymax - pred_ymin)
            true_area = (true_xmax - true_xmin) * (true_ymax - true_ymin)
            union = pred_area + true_area - intersection
            iou = intersection / union.clamp(min=1e-6)
            iou_scores.append(iou.mean().item())

    avg_train  = sum(train_losses)  / len(train_losses)
    avg_val    = sum(val_losses)    / len(val_losses)
    avg_iou    = sum(iou_scores)    / len(iou_scores)
    avg_px_err = sum(pixel_errors)  / len(pixel_errors)
    elapsed    = time.time() - start

    print(f"Epoch {epoch+1:02d}/50 | Train: {avg_train:.4f} | Val: {avg_val:.4f} | IoU: {avg_iou:.3f} | Pixel Error: {avg_px_err:.1f}px | Time: {elapsed:.1f}s")

    # Save best model based on IoU now
    if avg_iou > best_val_iou:
        best_val_iou = avg_iou
        torch.save(model.state_dict(), "best_model.pth")
        print(f"           ↑ saved new best (IoU={avg_iou:.3f}, pixel_err={avg_px_err:.1f}px)")

    scheduler.step()

Using: cuda
Found 1692 samples in data/train_data/train_data


Epoch 01/50 | Train: 3.4797 | Val: 0.9980 | IoU: 0.002 | Pixel Error: 403.8px | Time: 19.3s
           ↑ saved new best (IoU=0.002, pixel_err=403.8px)


Epoch 02/50 | Train: 2.2411 | Val: 0.9710 | IoU: 0.029 | Pixel Error: 253.0px | Time: 23.4s
           ↑ saved new best (IoU=0.029, pixel_err=253.0px)


Epoch 03/50 | Train: 1.5071 | Val: 0.9433 | IoU: 0.057 | Pixel Error: 271.6px | Time: 19.9s
           ↑ saved new best (IoU=0.057, pixel_err=271.6px)


Epoch 04/50 | Train: 1.2802 | Val: 0.8250 | IoU: 0.175 | Pixel Error: 115.6px | Time: 18.9s
           ↑ saved new best (IoU=0.175, pixel_err=115.6px)


...

Epoch 13/50 | Train: 9.0452 | Val: 0.7706 | IoU: 0.229 | Pixel Error: 530.6px | Time: 21.7s


Epoch 14/50 | Train: 8.9261 | Val: 0.7783 | IoU: 0.222 | Pixel Error: 528.2px | Time: 21.2s


Epoch 15/50 | Train: 8.8973 | Val: 0.7826 | IoU: 0.217 | Pixel Error: 533.8px | Time: 19.6s


Epoch 16/50 | Train: 8.8433 | Val: 0.7733 | IoU: 0.227 | Pixel Error: 530.5px | Time: 20.4s


KeyboardInterrupt: 

In [27]:
def iou_loss(pred, target, W, H):
    scale = torch.cat([W, H], dim=1).repeat(1, 4)
    pred_px = (pred * scale).view(-1, 4, 2)
    true_px = (target * scale).view(-1, 4, 2)

    pred_xmin = pred_px[:, :, 0].min(dim=1).values
    pred_ymin = pred_px[:, :, 1].min(dim=1).values
    pred_xmax = pred_px[:, :, 0].max(dim=1).values
    pred_ymax = pred_px[:, :, 1].max(dim=1).values

    true_xmin = true_px[:, :, 0].min(dim=1).values
    true_ymin = true_px[:, :, 1].min(dim=1).values
    true_xmax = true_px[:, :, 0].max(dim=1).values
    true_ymax = true_px[:, :, 1].max(dim=1).values

    inter_xmin = torch.max(pred_xmin, true_xmin)
    inter_ymin = torch.max(pred_ymin, true_ymin)
    inter_xmax = torch.min(pred_xmax, true_xmax)
    inter_ymax = torch.min(pred_ymax, true_ymax)

    inter_w = (inter_xmax - inter_xmin).clamp(min=0)
    inter_h = (inter_ymax - inter_ymin).clamp(min=0)
    intersection = inter_w * inter_h

    pred_area = (pred_xmax - pred_xmin) * (pred_ymax - pred_ymin)
    true_area = (true_xmax - true_xmin) * (true_ymax - true_ymin)
    union = pred_area + true_area - intersection

    iou = intersection / union.clamp(min=1e-6)
    return 1 - iou.mean()


# ── 4. TRAINING ───────────────────────────────────────────────────────────────

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

dataset = ShadowDataset("data/train_data/train_data")

train_size = int(0.8 * len(dataset))
val_size   = len(dataset) - train_size
train_set, val_set = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=32)

model     = ShadowDetector().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', 
                                                        patience=3, factor=0.5, 
                                                        verbose=True)
dir_loss  = nn.BCELoss()

best_val_iou = 0.0

for epoch in range(50):
    start = time.time()
    
    # Use MSE for first 10 epochs to get boxes in the right area
    # Then switch to IoU loss to fine tune overlap
    use_iou_loss = epoch >= 10

    # ── Train ──
    model.train()
    train_losses = []
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1:02d}/50 [Train]", leave=False)
    for imgs, corners, directions, W, H, _ in loop:
        imgs       = imgs.to(device)
        corners    = corners.to(device)
        directions = directions.to(device)
        W_s = W.float().to(device).unsqueeze(1)
        H_s = H.float().to(device).unsqueeze(1)
        
        optimizer.zero_grad()
        pred_corners, pred_dir = model(imgs)
        
        if use_iou_loss:
            box_l = 0.5 * iou_loss(pred_corners, corners, W_s, H_s) + \
            0.5 * nn.MSELoss()(pred_corners, corners)
        else:
            box_l = nn.MSELoss()(pred_corners, corners)

        loss = box_l * 10 + dir_loss(pred_dir, directions)
        
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())
        loop.set_postfix(loss=f"{loss.item():.4f}", mode="IoU" if use_iou_loss else "MSE")
    
    # ── Validate ──
    model.eval()
    val_losses  = []
    iou_scores  = []
    pixel_errors = []
    loop = tqdm(val_loader, desc=f"Epoch {epoch+1:02d}/50 [Val]  ", leave=False)
    with torch.no_grad():
        for imgs, corners, directions, W, H, _ in loop:
            imgs       = imgs.to(device)
            corners    = corners.to(device)
            directions = directions.to(device)
            W_s = W.float().to(device).unsqueeze(1)
            H_s = H.float().to(device).unsqueeze(1)
            
            pred_corners, pred_dir = model(imgs)

            loss = iou_loss(pred_corners, corners, W_s, H_s)
            val_losses.append(loss.item())
            loop.set_postfix(loss=f"{loss.item():.4f}")

            # Pixel error
            scale = torch.cat([W_s, H_s], dim=1).repeat(1, 4)
            pred_px = pred_corners * scale
            true_px = corners      * scale
            diff = (pred_px - true_px).view(-1, 4, 2)
            dist = torch.sqrt((diff ** 2).sum(dim=2))
            pixel_errors.append(dist.mean().item())

            # IoU
            pred_px = pred_px.view(-1, 4, 2)
            true_px = true_px.view(-1, 4, 2)

            pred_xmin = pred_px[:, :, 0].min(dim=1).values
            pred_ymin = pred_px[:, :, 1].min(dim=1).values
            pred_xmax = pred_px[:, :, 0].max(dim=1).values
            pred_ymax = pred_px[:, :, 1].max(dim=1).values

            true_xmin = true_px[:, :, 0].min(dim=1).values
            true_ymin = true_px[:, :, 1].min(dim=1).values
            true_xmax = true_px[:, :, 0].max(dim=1).values
            true_ymax = true_px[:, :, 1].max(dim=1).values

            inter_xmin = torch.max(pred_xmin, true_xmin)
            inter_ymin = torch.max(pred_ymin, true_ymin)
            inter_xmax = torch.min(pred_xmax, true_xmax)
            inter_ymax = torch.min(pred_ymax, true_ymax)

            inter_w = (inter_xmax - inter_xmin).clamp(min=0)
            inter_h = (inter_ymax - inter_ymin).clamp(min=0)
            intersection = inter_w * inter_h

            pred_area = (pred_xmax - pred_xmin) * (pred_ymax - pred_ymin)
            true_area = (true_xmax - true_xmin) * (true_ymax - true_ymin)
            union = pred_area + true_area - intersection
            iou = intersection / union.clamp(min=1e-6)
            iou_scores.append(iou.mean().item())

    avg_train  = sum(train_losses)  / len(train_losses)
    avg_val    = sum(val_losses)    / len(val_losses)
    avg_iou    = sum(iou_scores)    / len(iou_scores)
    avg_px_err = sum(pixel_errors)  / len(pixel_errors)
    elapsed    = time.time() - start

    print(f"Epoch {epoch+1:02d}/50 | Train: {avg_train:.4f} | Val: {avg_val:.4f} | IoU: {avg_iou:.3f} | Pixel Error: {avg_px_err:.1f}px | Time: {elapsed:.1f}s")

    # Save best model based on IoU now
    if avg_iou > best_val_iou:
        best_val_iou = avg_iou
        torch.save(model.state_dict(), "best_model.pth")
        print(f"           ↑ saved new best (IoU={avg_iou:.3f}, pixel_err={avg_px_err:.1f}px)")

    scheduler.step(avg_iou)

c:\Users\prosh\anaconda3\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Using: cuda
Found 1692 samples in data/train_data/train_data


Epoch 01/50 | Train: 3.7358 | Val: 1.0000 | IoU: 0.000 | Pixel Error: 462.1px | Time: 20.9s


Epoch 02/50 | Train: 2.7174 | Val: 0.9613 | IoU: 0.039 | Pixel Error: 301.0px | Time: 20.6s
           ↑ saved new best (IoU=0.039, pixel_err=301.0px)


Epoch 03/50 | Train: 1.8084 | Val: 0.9029 | IoU: 0.097 | Pixel Error: 160.2px | Time: 18.9s
           ↑ saved new best (IoU=0.097, pixel_err=160.2px)


...

Epoch 17/50 | Train: 4.6027 | Val: 0.7978 | IoU: 0.202 | Pixel Error: 143.8px | Time: 18.8s


Epoch 18/50 | Train: 4.5772 | Val: 0.6653 | IoU: 0.335 | Pixel Error: 86.9px | Time: 19.6s


Epoch 19/50 | Train: 4.6125 | Val: 0.6933 | IoU: 0.307 | Pixel Error: 85.7px | Time: 18.9s


Epoch 20/50 | Train: 4.7111 | Val: 0.7410 | IoU: 0.259 | Pixel Error: 111.0px | Time: 18.3s


KeyboardInterrupt: 